# Notebook 0: Environment Check - GitHub Codespaces

**Python Applied to Business**

This notebook verifies the preinstalled course environment. It does **not** install or upgrade libraries.

**Validated baseline provider:** Alibaba Cloud Model Studio, **Singapore**, using `qwen3.7-flash`.

**Important:** Before running the API tests, make sure **Free Quota Only / Stop-on-Exhaust** is enabled for `qwen3.7-flash` in Alibaba Cloud Model Studio.

Run the cells in order. Your API key is entered with a hidden prompt and is stored only in the current Python kernel session.

> GLM remains a possible alternative provider for the course. We will retest its onboarding and access from mainland China. It is not required for this environment check.


## Step 1: Verify the preinstalled environment

All course dependencies are installed automatically when the Codespace is created.

This test checks the principal packages and reports their installed versions. It makes no external API calls.


In [ ]:
import sys
import importlib
from importlib import metadata

checks = [
    ("LangChain", "langchain", "langchain"),
    ("LangChain OpenAI", "langchain_openai", "langchain-openai"),
    ("ChromaDB", "chromadb", "chromadb"),
    ("Sentence Transformers", "sentence_transformers", "sentence-transformers"),
    ("LangChain HuggingFace", "langchain_huggingface", "langchain-huggingface"),
    ("CrewAI", "crewai", "crewai"),
    ("CrewAI Tools", "crewai_tools", "crewai-tools"),
    ("OpenAI SDK", "openai", "openai"),
    ("LiteLLM", "litellm", "litellm"),
    ("PyPDF", "pypdf", "pypdf"),
    ("Unstructured", "unstructured", "unstructured"),
    ("Wikipedia", "wikipedia", "wikipedia"),
]

print(f"[OK] Python {sys.version.split()[0]}")

failed = []

for label, module_name, distribution_name in checks:
    try:
        importlib.import_module(module_name)
        try:
            version = metadata.version(distribution_name)
        except metadata.PackageNotFoundError:
            version = "installed"
        print(f"[OK] {label}: {version}")
    except Exception as exc:
        failed.append((label, exc))
        print(f"[FAIL] {label}: {exc}")

assert not failed, "One or more required packages could not be imported."
print("\n[OK] Core environment verified")


## Step 2: Enter your Alibaba Model Studio API key

The course baseline uses **Alibaba Cloud Model Studio, Singapore**.

The key is hidden while typing and is stored only in the current Python kernel session. It is **not** written into the notebook.

Do not paste an API key directly into a code cell.


In [ ]:
import os
from getpass import getpass

API_KEY_ENV = "DASHSCOPE_API_KEY"

if not os.getenv(API_KEY_ENV):
    os.environ[API_KEY_ENV] = getpass(
        "Enter your Alibaba Model Studio API Key: "
    ).strip()

assert os.environ[API_KEY_ENV], f"{API_KEY_ENV} cannot be empty"
print(f"[OK] {API_KEY_ENV} loaded for this notebook session")


## Step 3: Test the Qwen connection

This test calls `qwen3.7-flash` through Alibaba Model Studio's OpenAI-compatible endpoint in Singapore.

Deep thinking is explicitly disabled to keep token consumption and latency moderate.


In [ ]:
from openai import OpenAI

QWEN_MODEL = "qwen3.7-flash"
QWEN_BASE_URL = "https://dashscope-intl.aliyuncs.com/compatible-mode/v1"

qwen_client = OpenAI(
    api_key=os.environ["DASHSCOPE_API_KEY"],
    base_url=QWEN_BASE_URL,
)

response = qwen_client.chat.completions.create(
    model=QWEN_MODEL,
    messages=[
        {
            "role": "user",
            "content": "Reply exactly with: QWEN CONNECTION OK",
        }
    ],
    temperature=0,
    max_tokens=100,
    extra_body={"enable_thinking": False},
)

reply = response.choices[0].message.content.strip()

print("[OK] Qwen API call completed")
print("LLM replies:", reply)
print(
    "Token usage:",
    f"input={response.usage.prompt_tokens},",
    f"output={response.usage.completion_tokens},",
    f"total={response.usage.total_tokens}",
)

assert "QWEN CONNECTION OK" in reply.upper(), (
    "The API call succeeded, but the model did not return the expected test phrase."
)


## Step 4: Test local multilingual embeddings

Embeddings run locally inside the Codespace and consume **no LLM API quota**.

The model is preloaded during Codespace creation:

`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`

Progress display is disabled so the test does not depend on external JavaScript widget CDNs.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    show_progress=False,
)

vector = embeddings.embed_query("This is an embeddings test for RAG")

print("[OK] Local embeddings work correctly")
print("Vector dimension:", len(vector))
print("First 5 values:", vector[:5])

assert len(vector) == 384, (
    f"Unexpected embedding dimension: {len(vector)}; expected 384."
)


## Step 5: Test ChromaDB using the same local embeddings

This test explicitly supplies the local MiniLM embeddings to ChromaDB.

That avoids any hidden download or use of ChromaDB's default embedding model.


In [ ]:
import uuid
import chromadb

client = chromadb.Client()
collection_name = f"course_environment_test_{uuid.uuid4().hex[:8]}"

try:
    collection = client.create_collection(collection_name)

    document = "Python is a versatile programming language"
    document_embedding = embeddings.embed_query(document)

    collection.add(
        documents=[document],
        embeddings=[document_embedding],
        ids=["doc1"],
    )

    query_embedding = embeddings.embed_query("What is Python?")
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=1,
    )

    retrieved = results["documents"][0][0]

    print("[OK] ChromaDB works correctly")
    print("Query result:", retrieved)

    assert retrieved == document, "Unexpected document returned by ChromaDB."

finally:
    try:
        client.delete_collection(collection_name)
    except Exception:
        pass


## Step 6: Check optional / legacy LLM integrations

These integrations are retained for compatibility with existing course notebooks.

This cell only checks that the Python classes can be imported. It does **not** make external API calls.

- Groq
- OpenAI
- Google Gemini

GLM does not require a dedicated package for our intended use because it also exposes an OpenAI-compatible API.


In [ ]:
import importlib

integrations = [
    ("Groq", "langchain_groq", "ChatGroq"),
    ("OpenAI", "langchain_openai", "ChatOpenAI"),
    ("Google Gemini", "langchain_google_genai", "ChatGoogleGenerativeAI"),
]

available = []
failed = []

for label, module_name, class_name in integrations:
    try:
        module = importlib.import_module(module_name)
        getattr(module, class_name)
        available.append(label)
        print(f"[OK] {label}")
    except Exception as exc:
        failed.append((label, exc))
        print(f"[FAIL] {label}: {exc}")

assert not failed, "One or more compatibility integrations are unavailable."
print("\n[OK] Optional / legacy integrations installed:", ", ".join(available))


## Step 7: Test CrewAI with Qwen

This is the minimal end-to-end agent test:

`CrewAI -> Agent -> Task -> qwen3.7-flash -> result`

Because this notebook runs inside Jupyter, CrewAI must be launched with:

`await crew.kickoff_async()`

Using synchronous `crew.kickoff()` in this environment can collide with Jupyter's running event loop.


In [ ]:
from crewai import Agent, Task, Crew, LLM

crew_llm = LLM(
    model=QWEN_MODEL,
    custom_openai=True,
    base_url=QWEN_BASE_URL,
    api_key=os.environ["DASHSCOPE_API_KEY"],
    temperature=0,
    max_tokens=256,
    extra_body={"enable_thinking": False},
)

test_agent = Agent(
    role="Verifier",
    goal="Confirm that CrewAI works",
    backstory="You are a minimal environment test agent.",
    llm=crew_llm,
    verbose=False,
    allow_delegation=False,
    max_iter=2,
)

test_task = Task(
    description="Reply exactly: CrewAI works correctly.",
    expected_output="CrewAI works correctly.",
    agent=test_agent,
)

crew = Crew(
    agents=[test_agent],
    tasks=[test_task],
    verbose=False,
)

result = await crew.kickoff_async()

print("[OK] CrewAI executed successfully")
print("Result:", str(result)[:200])
print("Token usage:", crew.usage_metrics)

reasoning_tokens = getattr(crew.usage_metrics, "reasoning_tokens", 0)
assert reasoning_tokens == 0, (
    f"Unexpected reasoning-token usage: {reasoning_tokens}. "
    "Check that enable_thinking=False is still being applied."
)

print("[OK] Qwen thinking is disabled")


## Step 8: Test CrewAI tool calling

This final test verifies that Qwen can select and call a Python tool through CrewAI.

It normally requires two model requests: one to select the tool and one to produce the final answer.


In [ ]:
from crewai.tools import tool

@tool("Calculate profit margin")
def calculate_profit_margin(revenue: float, cost: float) -> str:
    """Calculate profit and profit margin percentage from revenue and cost."""
    profit = revenue - cost
    margin = (profit / revenue) * 100
    return f"Profit: {profit:.2f}; Profit margin: {margin:.2f}%"

tool_agent = Agent(
    role="Financial Analyst",
    goal="Perform accurate business calculations using the available tools.",
    backstory=(
        "You are a financial analyst. When a calculation tool is available, "
        "you must use it rather than calculating the result yourself."
    ),
    llm=crew_llm,
    tools=[calculate_profit_margin],
    verbose=False,
    allow_delegation=False,
    max_iter=3,
)

tool_task = Task(
    description=(
        "A company has revenue of 250000 euros and costs of 175000 euros. "
        "Use the available tool to calculate its profit and profit margin."
    ),
    expected_output="The profit in euros and the profit margin percentage.",
    agent=tool_agent,
)

tool_crew = Crew(
    agents=[tool_agent],
    tasks=[tool_task],
    verbose=False,
)

tool_result = await tool_crew.kickoff_async()

print("[OK] CrewAI tool calling executed successfully")
print("Result:", str(tool_result)[:250])
print("Token usage:", tool_crew.usage_metrics)

reasoning_tokens = getattr(tool_crew.usage_metrics, "reasoning_tokens", 0)
assert reasoning_tokens == 0, (
    f"Unexpected reasoning-token usage: {reasoning_tokens}. "
    "Check that enable_thinking=False is still being applied."
)

print("[OK] Tool calling works and Qwen thinking remains disabled")


## Verification summary

If every previous cell completed without errors, the Codespace baseline is ready for testing with the real course notebooks.

| Component | Expected status |
|---|---|
| Python and preinstalled libraries | OK |
| Alibaba Model Studio API key | OK |
| Qwen `qwen3.7-flash` connection | OK |
| Thinking disabled | OK |
| Local multilingual embeddings | OK |
| ChromaDB with explicit local embeddings | OK |
| Optional / legacy LLM integrations | OK |
| CrewAI + Qwen | OK |
| CrewAI tool calling | OK |
| Token usage metrics | OK |

### Important course conventions

1. In Jupyter notebooks, launch crews with `await crew.kickoff_async()`.
2. Keep Qwen deep thinking disabled unless a specific exercise requires it.
3. Do not hardcode API keys in notebooks.
4. Do not install or upgrade packages manually inside the student Codespace.
5. If a test fails, stop and report the exact error before changing the environment.

### Provider note

Qwen on Alibaba Cloud Model Studio Singapore is the **currently validated baseline**.

GLM remains a candidate alternative. We will retest GLM from mainland China, particularly if students can authenticate with Chinese mobile numbers or other locally supported onboarding methods.
